# **UNIVERSIDAD PRIVADA ANTENOR ORREGO**
## **FACULTAD DE INGENIERÍA**
### **INGENIERÍA DE SISTEMAS E INTELIGENCIA ARTIFICIAL**

**CURSO:** BIG DATA Y ANALÍTICA DE DATOS  
**DOCENTE:** Ing. Luis Nicolas Castillo Ramirez Keller  
**SESIÓN 12:** PROCESAMIENTO DE DATOS USANDO APACHE SPARK (Laboratorio S12)  
**FECHA:** 27/06/2026

---
### **Carátula del Estudiante:**
- **Nombres y Apellidos:** Vanesa Marili Vera Romero
- **ID de Estudiante:** 000269782

---

# **PARTE 1: LABORATORIO GUIADO**

Este caso práctico se centra en la simulación y el procesamiento de datos de sensores IoT en tiempo real utilizando Apache Spark Structured Streaming en Google Colab. El objetivo es demostrar cómo Spark puede manejar flujos de datos continuos y proporcionar un monitoreo en vivo.

Se realizarán los siguientes pasos:
1.  **Preparación del Entorno:** Se instalará PySpark y se inicializará una sesión de Spark. También se creará una carpeta local que simulará un "servidor de ingesta" para los datos de los sensores.
2.  **Definición de Esquema y Lectura de Stream:** Se definirá un esquema estricto (`StructType`) para los datos entrantes (ID del sensor y temperatura). Spark se configurará para "escuchar" la carpeta de ingesta, procesando los archivos JSON a medida que llegan.
3.  **Lógica de Negocio:** Se aplicará una lógica simple para calcular la temperatura máxima por cada sensor (`sensor_id`).
4.  **Arranque del Flujo y Almacenamiento en Memoria:** Los resultados agregados (temperaturas máximas por sensor) se escribirán continuamente en una tabla temporal en la memoria de Colab (`format("memory")`) utilizando el modo `outputMode("complete")`.
5.  **Simulación y Dashboard en Vivo:** Un simulador de Python generará archivos JSON con datos de sensores aleatorios y los depositará en la carpeta de ingesta. Simultáneamente, se consultará la tabla en memoria y se actualizará dinámicamente la salida de la celda (`clear_output()`) para simular un dashboard de monitoreo en tiempo real de las temperaturas máximas de cada sensor.

Usar Google Colab para Structured Streaming es un excelente movimiento técnico, pero presenta un desafío visual: a diferencia de Databricks, Colab no tiene gráficos que se actualicen solos de forma nativa.

Para solucionar esto, aplicaremos un truco avanzado de Python usando IPython.display. Borraremos y reescribiremos la salida de la celda en cada iteración para simular un Dashboard en tiempo real.

### **Celda 1. Preparación del Entorno y Sesión**
**Objetivo:** Instalar PySpark en la máquina virtual, inicializar la sesión de Spark y crear/limpiar una carpeta local (`/content/iot_stream/`). Esta carpeta actuará como nuestro "servidor de ingesta", donde llegarán los archivos en tiempo real.

In [1]:
# Configuración Inicial en Colab
!pip install -q pyspark

from pyspark.sql import SparkSession
import os
import shutil

# 1. Iniciar Spark en la máquina virtual de Google
spark = SparkSession.builder.appName("Streaming_IoT_Colab").getOrCreate()

# 2. Crear las carpetas locales en Colab para simular el servidor
ruta_streaming = "/content/iot_stream/"

# Si la carpeta ya existe de una ejecución previa, la borramos para empezar limpio
if os.path.exists(ruta_streaming):
    shutil.rmtree(ruta_streaming)
os.makedirs(ruta_streaming)

print(f"Entorno listo. Carpeta de escucha creada en: {ruta_streaming}")

Entorno listo. Carpeta de escucha creada en: /content/iot_stream/


### **Celda 2. Esquema Estricto y Motor de Lectura (ReadStream)**
**Objetivo:** Definir el `StructType` de los datos entrantes. A diferencia del procesamiento Batch, en Streaming **es obligatorio declarar el esquema** para evitar que un archivo corrupto tumbe el sistema. Luego, ponemos a Spark a "escuchar" la carpeta y aplicamos la lógica de negocio al vuelo: calcular la temperatura máxima por sensor.

In [2]:
# Definir Esquema y Lectura (ReadStream)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import max

# En streaming, declarar el esquema es OBLIGATORIO
esquema_iot = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("temperatura", DoubleType(), True)
])

# Conectar Spark a la carpeta
df_stream = spark.readStream.schema(esquema_iot).json(ruta_streaming)

# Lógica de negocio: Temperatura máxima por máquina
df_agregado = df_stream.groupBy("sensor_id").agg(max("temperatura").alias("temp_maxima"))

### **Celda 3: Arranque del Flujo (WriteStream)**
**Objetivo:** Encender el motor de procesamiento continuo. Configuramos la salida hacia la memoria RAM de Colab (`format("memory")`) usando el `outputMode("complete")`, lo que significa que la tabla temporal se reescribirá por completo cada vez que un nuevo micro-lote de datos actualice los valores máximos.

In [3]:
# Ejecución y Almacenamiento en Memoria (WriteStream)

query = (df_agregado.writeStream
         .format("memory") # Guardamos en la RAM de Colab
         .queryName("tabla_iot_vivo") # Nombre de la tabla temporal
         .outputMode("complete") # Modo Complete para ver los agregados totales
         .start())

print("Stream iniciado. Escuchando archivos nuevos...")

Stream iniciado. Escuchando archivos nuevos...


### **Celda 4: Simulador IoT y Dashboard en Vivo**
**Objetivo:** Como no tenemos sensores físicos conectados, este bucle de Python generará pequeños archivos JSON con temperaturas aleatorias y los depositará en nuestra carpeta cada 2 segundos. Simultáneamente, consultaremos la tabla en memoria y limpiaremos la pantalla para simular un **Dashboard de monitoreo en tiempo real**.

In [4]:
# Simulador + Dashboard en Vivo (La magia en Colab)
import time
import random
from IPython.display import clear_output

sensores = ["Maquina_A", "Maquina_B", "Maquina_C"]

print("Iniciando inyección de datos y monitoreo...")
time.sleep(2)

# Simularemos 15 envíos de datos
for i in range(15):
    # 1. Crear un archivo JSON simulando la red
    sensor = random.choice(sensores)
    temp = round(random.uniform(50.0, 120.0), 2)
    contenido = f'{{"sensor_id": "{sensor}", "temperatura": {temp}}}'

    with open(f"{ruta_streaming}lectura_{i}.json", "w") as f:
        f.write(contenido)

    # 2. Darle 2 segundos a Spark para leerlo y procesarlo
    time.sleep(2)

    # 3. Actualizar la pantalla (Dashboard)
    clear_output(wait=True)
    print(f"🔴 DASHBOARD EN VIVO - Micro-lote {i+1}/15")
    spark.sql("SELECT * FROM tabla_iot_vivo ORDER BY sensor_id").show()

# Detenemos el motor de streaming al terminar la simulación
query.stop()
print("Simulación finalizada.")

🔴 DASHBOARD EN VIVO - Micro-lote 15/15
+---------+-----------+
|sensor_id|temp_maxima|
+---------+-----------+
|Maquina_A|      116.1|
|Maquina_B|     109.39|
|Maquina_C|     113.58|
+---------+-----------+

Simulación finalizada.


---
# **PARTE 2: TAREA AUTÓNOMA - Procesamiento en Tiempo Real con Spark Structured Streaming S12**

### **Instrucciones para el Alumno:**

**Contexto:** Ciberseguridad y Monitoreo de Tráfico. Eres el encargado de vigilar la salud de la infraestructura web de la empresa.

**El Reto de Automatización:** En el mundo real, los logs de los servidores llegan constantemente. Utilizarás un "Simulador de Tráfico" (código provisto) que generará alertas y pequeños archivos CSV en vivo, inyectándolos en una carpeta. Tu objetivo es procesar esos archivos a medida que nacen usando Big Data.

### **Pasos Obligatorios a Codificar:**
- El Spooler (Simulador provisto): Ejecutar la celda entregada por el docente. Esta creará la carpeta `/content/logs_web/` e inyectará continuamente pequeños archivos CSV (con las columnas `servidor` y `codigo_http`).

- Misión de PySpark:
  - Declarar el `StructType` exacto para leer los archivos CSV generados (una columna de texto y una numérica).

  - Usar .readStream apuntando a `/content/logs_web/`.

  - Filtrar la data en vivo para conservar solo los registros que tengan `codigo_http >= 402`.

  - Agrupar y contar los errores por servidor (`.groupBy("servidor").count()`).

- Visualización: Arrancar el `.writeStream` hacia la memoria (`format("memory")`) usando el `outputMode("complete")`. Implementar un bucle con `clear_output()` para consultar la tabla temporal y mostrar en vivo en la pantalla qué servidor está colapsando.
- Añadir interpretación de conclusiones y la importancia de Apache Spark Structured Streaming en un entorno real citando un par de ejemplos relacionados a su proyecto.

**ENTREGA:** Subir este archivo `.ipynb` con su nombre en la carátula y el código de la Tarea al final de este mismo archivo.

---

## **Desarrollo del Laboratorio:**

El Código "Spooler":
(Adaptar esta celda en las celdas 1 y 4 respectivamente. Este script inventará el tráfico y los errores en tiempo real.)

In [9]:
# ==========================================
# SIMULADOR DE TRÁFICO WEB SINTÉTICO
# ==========================================
import os
import time
import random

## Celda 1: Configuración Inicial y Preparación del Entorno
# Creamos la carpeta donde llegarán los datos en vivo
ruta_logs = "/content/logs_web/"
os.makedirs(ruta_logs, exist_ok=True)

## Celda 4: Simulador de Tráfico Web y Dashboard en Vivo
servidores = ["Servidor_Delta", "Servidor_Epsilon", "Servidor_Zeta"]
# Simulamos mayoritariamente tráfico sano (200) y algunos errores (402, 403, 404, 500)
codigos = [200, 200, 200, 200, 200, 402, 403, 404, 500]

print("Iniciando inyección de tráfico web en vivo...")

# El generador creará 20 lotes de datos
for i in range(20):
    lineas = []
    # Cada lote tendrá 15 registros simulados
    for _ in range(15):
        servidor = random.choice(servidores)
        codigo = random.choice(codigos)
        lineas.append(f"{servidor},{codigo}\n")

    # Escribimos el micro-lote en un archivo CSV sin cabeceras
    with open(f"{ruta_logs}log_stream_{i}.csv", "w") as f:
        f.writelines(lineas)

    print(f"Tráfico inyectado: Micro-lote {i+1}/20 generado con éxito.")
    time.sleep(3) # Pausa de 3 segundos para darle tiempo a Spark de procesar

Iniciando inyección de tráfico web en vivo...
Tráfico inyectado: Micro-lote 1/20 generado con éxito.
Tráfico inyectado: Micro-lote 2/20 generado con éxito.
Tráfico inyectado: Micro-lote 3/20 generado con éxito.
Tráfico inyectado: Micro-lote 4/20 generado con éxito.
Tráfico inyectado: Micro-lote 5/20 generado con éxito.
Tráfico inyectado: Micro-lote 6/20 generado con éxito.
Tráfico inyectado: Micro-lote 7/20 generado con éxito.
Tráfico inyectado: Micro-lote 8/20 generado con éxito.
Tráfico inyectado: Micro-lote 9/20 generado con éxito.
Tráfico inyectado: Micro-lote 10/20 generado con éxito.
Tráfico inyectado: Micro-lote 11/20 generado con éxito.
Tráfico inyectado: Micro-lote 12/20 generado con éxito.
Tráfico inyectado: Micro-lote 13/20 generado con éxito.
Tráfico inyectado: Micro-lote 14/20 generado con éxito.
Tráfico inyectado: Micro-lote 15/20 generado con éxito.
Tráfico inyectado: Micro-lote 16/20 generado con éxito.
Tráfico inyectado: Micro-lote 17/20 generado con éxito.
Tráfico iny

---

### **Celda 1: Configuración Inicial y Preparación del Entorno**
**Objetivo:** Crear la carpeta del Workspace donde se depositarán los registros en tiempo real e inicializar las librerías necesarias.

In [6]:
import os
import shutil
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Definir la ruta donde el simulador va a inyectar los archivos CSV
ruta_logs = "logs_web/"

# 2. Empezar con la carpeta limpia si existía una ejecución previa
if os.path.exists(ruta_logs):
    shutil.rmtree(ruta_logs)
os.makedirs(ruta_logs)

print(f"Entorno listo. Carpeta de escucha de logs creada en: {ruta_logs}")

Entorno listo. Carpeta de escucha de logs creada en: logs_web/


### **Celda 2: Esquema Estricto y Lógica de Filtrado (ReadStream)**
**Objetivo:** Declarar de forma rígida el formato del CSV (servidor y código HTTP) para proteger el flujo continuo y construir la lógica para capturar solo errores `(codigo_http >= 402)`.

In [7]:
from pyspark.sql.functions import col

# 1. En Structured Streaming es OBLIGATORIO definir el esquema para evitar caídas del sistema
esquema_logs = StructType([
    StructField("servidor", StringType(), True),
    StructField("codigo_http", IntegerType(), True)
])

# 2. Conectar el motor de escucha al directorio en tiempo real
df_stream_logs = spark.readStream \
    .schema(esquema_logs) \
    .csv(ruta_logs)

# 3. Lógica de ciberseguridad: Filtrar errores (>= 402) y agrupar conteos por servidor
df_errores_agregados = df_stream_logs \
    .filter(col("codigo_http") >= 402) \
    .groupBy("servidor") \
    .count()

print("Estructura de ReadStream y lógica de filtrado configurada con éxito.")

Estructura de ReadStream y lógica de filtrado configurada con éxito.


### **Celda 3: Arranque del Flujo Continuo (WriteStream)**
**Objetivo:** Encender el motor de procesamiento en tiempo real guardando el consolidado dinámico en la RAM para poder analizarlo.

In [8]:
# Encender el stream de escritura en modo completo para actualizar la tabla continuamente
query_logs = df_errores_agregados.writeStream \
    .format("memory") \
    .queryName("tabla_errores_vivo") \
    .outputMode("complete") \
    .start()

print("Motor de streaming encendido. Esperando inyección de archivos...")

Motor de streaming encendido. Esperando inyección de archivos...


### **Celda 4: Simulador de Tráfico Web y Dashboard en Vivo**
**Objetivo:** Ejecutar la inyección programada de los 20 micro-lotes de archivos CSV directamente en tu directorio y mostrar la tabla de control en vivo.

In [10]:
import time
import random

servidores = ["Servidor_Delta", "Servidor_Epsilon", "Servidor_Zeta"]
# 200 representa peticiones sanas; códigos >= 402 representan fallas operativas
codigos_pool = [200, 200, 200, 200, 200, 402, 403, 404, 500]

print("Iniciando inyección de tráfico web sintético...")

# El bucle generará 20 archivos secuenciales cada 3 segundos
for i in range(20):
    lineas = []
    for _ in range(15):
        srv = random.choice(servidores)
        cod = random.choice(codigos_pool)
        lineas.append(f"{srv},{cod}\n")

    # Escribir el lote en disco
    with open(f"{ruta_logs}log_stream_{i}.csv", "w") as f:
        f.writelines(lineas)

    print(f" -> Lote {i+1}/20 depositado en la carpeta.")
    time.sleep(3)

    # Mostrar el monitoreo en vivo consumiendo la tabla de memoria mediante la API de DataFrames
    print(f"🔴 MONITOR DE INFRAESTRUCTURA WEB EN VIVO (Iteración {i+1})")
    spark.table("tabla_errores_vivo").orderBy("servidor").show()

# Detener el flujo para liberar recursos del clúster al terminar la simulación
query_logs.stop()
print("Simulación finalizada de forma exitosa.")

Iniciando inyección de tráfico web sintético...
 -> Lote 1/20 depositado en la carpeta.
🔴 MONITOR DE INFRAESTRUCTURA WEB EN VIVO (Iteración 1)
+----------------+-----+
|        servidor|count|
+----------------+-----+
|  Servidor_Delta|   35|
|Servidor_Epsilon|   48|
|   Servidor_Zeta|   36|
+----------------+-----+

 -> Lote 2/20 depositado en la carpeta.
🔴 MONITOR DE INFRAESTRUCTURA WEB EN VIVO (Iteración 2)
+----------------+-----+
|        servidor|count|
+----------------+-----+
|  Servidor_Delta|   43|
|Servidor_Epsilon|   50|
|   Servidor_Zeta|   39|
+----------------+-----+

 -> Lote 3/20 depositado en la carpeta.
🔴 MONITOR DE INFRAESTRUCTURA WEB EN VIVO (Iteración 3)
+----------------+-----+
|        servidor|count|
+----------------+-----+
|  Servidor_Delta|   44|
|Servidor_Epsilon|   52|
|   Servidor_Zeta|   42|
+----------------+-----+

 -> Lote 4/20 depositado en la carpeta.
🔴 MONITOR DE INFRAESTRUCTURA WEB EN VIVO (Iteración 4)
+----------------+-----+
|        servidor|c

### **Conclusiones:**
...
